### Environment Variable

In [1]:
from types import SimpleNamespace
args = SimpleNamespace(
    dataset_name="FiscalNote/billsum",
    dataset_config_name=None,
    dataset_text_field=None,
    streaming=False,
    tokenizer_name="gpt2",
    output_dir="outputs/retnet-billsum",
    block_size=512,
    max_train_samples=None,
    max_eval_samples=1025,
    validation_split_percentage=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    weight_decay=0.01,
    num_train_epochs=3,
    max_steps=-1,
    warmup_ratio=0.03,
    logging_steps=20,
    save_steps=200,
    eval_steps=200,
    seed=42,
    bf16=False,
    fp16=False,
    push_to_hub=False,
    hub_model_id=None,
    hidden_size=512,
    intermediate_size=2048,
    num_hidden_layers=8,
    num_retention_heads=8,
    value_dim=None,
    dropout=0.1,
    activation_dropout=0.0,
    drop_path_rate=0.0,
    recurrent_chunk_size=128,
    chunkwise_recurrent=False,
)

In [2]:
#!/usr/bin/env python
from __future__ import annotations

import argparse
import logging
import math
import os
from functools import partial

from datasets import load_dataset
import transformers
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

from retnet_hf import RetNetConfig, RetNetForCausalLM
logger = logging.getLogger(__name__)

def infer_text_field(dataset_name: str, dataset_text_field: str | None) -> str | None:
    if dataset_text_field:
        return dataset_text_field
    lower = dataset_name.lower()
    if "billsum" in lower:
        return None  # handled by formatter below
    if "c4" in lower:
        return "text"
    return "text"

def load_raw_datasets(args: argparse.Namespace):
    lower = args.dataset_name.lower()
    if "billsum" in lower:
        ds = load_dataset(args.dataset_name, args.dataset_config_name)
        if "validation" in ds:
            train_ds, eval_ds = ds["train"], ds["validation"]
        else:
            if args.streaming:
                raise ValueError("Streaming Billsum validation split creation is not supported in this example.")
            split = ds["train"].train_test_split(
                test_size=args.validation_split_percentage / 100.0,
                seed=args.seed,
            )
            train_ds, eval_ds = split["train"], split["test"]
        return train_ds, eval_ds

    if "c4" in lower:
        config_name = args.dataset_config_name or "en"
        train_ds = load_dataset(args.dataset_name, config_name, split="train", streaming=args.streaming)
        eval_ds = load_dataset(args.dataset_name, config_name, split="validation", streaming=args.streaming)
        return train_ds, eval_ds

    ds = load_dataset(args.dataset_name, args.dataset_config_name)
    if "validation" in ds:
        return ds["train"], ds["validation"]
    if args.streaming:
        raise ValueError("Streaming is only wired here for datasets that already expose validation splits.")
    split = ds["train"].train_test_split(
        test_size=args.validation_split_percentage / 100.0,
        seed=args.seed,
    )
    return split["train"], split["test"]

## Append "text: " and "summary: " to the dataset to shove into causal_lm.
def format_for_causal_lm(example, dataset_name: str, text_field: str | None):
    lower = dataset_name.lower()
    if "billsum" in lower:
        #print ("billsum dataset detected")
        return {
            "text": f"Document:\n{example['text']}\n\nSummary:\n{example['summary']}"
        }
    if text_field is None:
        raise ValueError("Could not infer dataset text field. Pass --dataset_text_field.")
    return {"text": example[text_field]}

## Simply running the tokenizer
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["text"])

## Turn dataset into a continous stream, divided by blocks.
def group_texts(examples, block_size: int):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = list(result["input_ids"])
    return result

In [3]:
#args = parse_args()
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    level=logging.INFO,
)
set_seed(args.seed)

In [4]:
text_field = infer_text_field(args.dataset_name, args.dataset_text_field)
train_raw, eval_raw = load_raw_datasets(args)

#GET SUBSET ONLY
train_raw = train_raw.select(range(min(200, len(train_raw)))) 
eval_raw = eval_raw.select(range(min(20, len(train_raw))))

tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name, use_fast=True)
tokenizer.model_max_length = 512
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

format_fn = partial(format_for_causal_lm, dataset_name=args.dataset_name, text_field=text_field)

remove_columns = getattr(train_raw, "column_names", None)
train_ds = train_raw.map(format_fn, remove_columns=remove_columns)
eval_ds = eval_raw.map(format_fn, remove_columns=getattr(eval_raw, "column_names", None))

#print (train_ds[0])

tokenize_fn = partial(tokenize_function, tokenizer=tokenizer)
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

#print(tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False))

eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

if args.max_train_samples is not None and not args.streaming:
    train_ds = train_ds.select(range(min(args.max_train_samples, len(train_ds))))
if args.max_eval_samples is not None and not args.streaming:
    eval_ds = eval_ds.select(range(min(args.max_eval_samples, len(eval_ds))))

group_fn = partial(group_texts, block_size=args.block_size)
train_ds = train_ds.map(group_fn, batched=True)
eval_ds = eval_ds.map(group_fn, batched=True)
print(tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False))

2026-05-07 01:20:41,297 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum "HTTP/1.1 200 OK"
2026-05-07 01:20:41,397 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum "HTTP/1.1 200 OK"
2026-05-07 01:20:41,639 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum/revision/3d8510441c06a3d9dfb32eb0d7f80151730bcc4f "HTTP/1.1 200 OK"
2026-05-07 01:20:41,676 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum/tree/3d8510441c06a3d9dfb32eb0d7f80151730bcc4f/data?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-07 01:20:41,698 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum/tree/3d8510441c06a3d9dfb32eb0d7f80151730bcc4f?recursive=false&expand=false "HTTP/1.1 200 OK"
2026-05-07 01:20:41,872 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-

Document:
SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention 
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal 
year does not become law prior to the beginning of these fiscal years 
or a joint resolution making continuing appropriations is not in 
effect, there is appropriated, out of any moneys in the Treasury not 
otherwise appropriated, and out of applicable corporate or other 
revenues, receipts, and funds, such sums as may be necessary to 
continue any program, project, or activity for which funds were 
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made available, and 
authority granted, for a program, project, or activity for any fiscal 
year pursuant to this Act shall be at 100 percent of the rate of 
operations that was provided for the program, project, or activity in 
fiscal year 1999 in the corresponding regular appropriation Act for 
fisca

In [5]:
config = RetNetConfig(
    vocab_size=len(tokenizer),
    hidden_size=args.hidden_size,
    intermediate_size=args.intermediate_size,
    num_hidden_layers=args.num_hidden_layers,
    num_retention_heads=args.num_retention_heads,
    value_dim=args.value_dim,
    hidden_dropout_prob=args.dropout,
    activation_dropout=args.activation_dropout,
    drop_path_rate=args.drop_path_rate,
    recurrent_chunk_size=args.recurrent_chunk_size,
    chunkwise_recurrent=args.chunkwise_recurrent,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
model = RetNetForCausalLM(config)
model.resize_token_embeddings(len(tokenizer))

Embedding(50257, 512)

In [6]:
training_args = TrainingArguments(
    output_dir=args.output_dir,
    #overwrite_output_dir=True,
    do_train=True,
    do_eval=True,
    eval_strategy ="steps",
    eval_steps=args.eval_steps,
    save_steps=args.save_steps,
    logging_steps=args.logging_steps,
    per_device_train_batch_size=args.per_device_train_batch_size,
    per_device_eval_batch_size=args.per_device_eval_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    learning_rate=args.learning_rate,
    weight_decay=args.weight_decay,
    num_train_epochs=args.num_train_epochs,
    max_steps=args.max_steps,
    warmup_ratio=args.warmup_ratio,
    lr_scheduler_type="cosine",
    bf16=args.bf16,
    fp16=args.fp16,
    push_to_hub=args.push_to_hub,
    hub_model_id=args.hub_model_id,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [7]:
transformers.logging.set_verbosity_info()

In [8]:
train_result = trainer.train("outputs/retnet-billsum/checkpoint-8567")
#train_result = trainer.train()
model = trainer.model
trainer.save_model(args.output_dir)
tokenizer.save_pretrained(args.output_dir)

metrics = trainer.evaluate()
if "eval_loss" in metrics:
    metrics["eval_perplexity"] = math.exp(metrics["eval_loss"])

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)
trainer.save_state()

print("Saved model to", os.path.abspath(args.output_dir))
print("Eval metrics:", metrics)

[transformers] Loading model from outputs/retnet-billsum/checkpoint-8567.
[transformers] There were missing keys in the checkpoint model loaded: ['model.decoder.embed_tokens.weight', 'model.decoder.output_projection.weight', 'lm_head.weight'].
[transformers] ***** Running training *****
[transformers]   Num examples = 1,513
[transformers]   Num Epochs = 3
[transformers]   Num update steps per epoch = 95
[transformers]   Instantaneous batch size per device = 2
[transformers]   Total train batch size (w. parallel, distributed & accumulation) = 16
[transformers]   Gradient Accumulation steps = 8
[transformers]   Total optimization steps = 285
[transformers]   Number of trainable parameters = 61,391,872
[transformers]   Resuming training from checkpoint with epoch 90 and global step 8567
[transformers]   Fast-forwarding the dataloader past 90 epochs and 136 batches to resume from the exact training state.
[transformers] 

Training completed. Do not forget to share your model on huggingface

Step,Training Loss,Validation Loss


[transformers] Saving model checkpoint to outputs/retnet-billsum
[transformers] Configuration saved in outputs/retnet-billsum\config.json
[transformers] Configuration saved in outputs/retnet-billsum\generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in outputs/retnet-billsum\model.safetensors
[transformers] tokenizer config file saved in outputs/retnet-billsum\tokenizer_config.json
[transformers] tokenizer config file saved in outputs/retnet-billsum\tokenizer_config.json
[transformers] 
***** Running Evaluation *****
[transformers]   Num examples = 149
[transformers]   Batch size = 2
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Training Loss,Validation Loss,Step
10.129729,1.285213,8567


***** train metrics *****
  epoch                    =         1.0
  total_flos               =  14006182GF
  train_loss               =         0.0
  train_runtime            =  0:00:00.00
  train_samples_per_second = 2269664.504
  train_steps_per_second   =  142510.329
***** eval metrics *****
  eval_loss       = 1.2852
  eval_perplexity = 3.6154
Saved model to C:\Users\Za\Desktop\SchoolWork\GMU\757 GAN\retnet_hf_wrapper\scripts\outputs\retnet-billsum
Eval metrics: {'eval_loss': 1.2852126359939575, 'eval_perplexity': 3.615436647454067}


In [9]:
prompt = """SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention 
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal 
year does not become law prior to the beginning of these fiscal years 
or a joint resolution making continuing appropriations is not in 
effect, there is appropriated, out of any moneys in the Treasury not 
otherwise appropriated, and out of applicable corporate or other 
revenues, receipts, and funds, such sums as may be necessary to 
continue any program, project, or activity for which funds were 
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made available, and 
authority granted, for a program, project, or activity for any fiscal 
year pursuant to this Act shall be at 100 percent of the rate of 
operations that was provided for the program, project, or activity in 
fiscal year 1999 in the corresponding regular appropriation Act for 
fiscal year 1999.
    (c) Period of Availability.--Appropriations and funds made 
available, and authority granted, for any fiscal year pursuant to this 
Act for a program, project, or activity shall be available for the 
period beginning with the first day of a lapse in appropriations and 
ending with the earlier of--
            (1) the date on which the applicable regular appropriation 
        bill for any fiscal year becomes law (whether or not that law 
        provides for that program, project, or activity) or a 
        continuing resolution making appropriations becomes law, as the 
        case may be; or
            (2) the last day of any fiscal year.

SEC. 3. TERMS AND CONDITIONS.

    (a) In General.--An appropriation of funds made available, or 
authority granted, for a program, project, or activity for any fiscal 
year pursuant to this Act shall be made available to the extent and in 
the manner which would be provided by the pertinent appropriations Act 
for fiscal year 1999, including all of the terms and conditions and the 
apportionment schedule imposed with respect to the appropriation made 
or funds made available for fiscal year 1999 or authority granted for 
the program, project, or activity under current law.
    (b) Extent and Manner.--Appropriations made by this Act shall be 
available to the extent and in the manner which would be provided by 
the pertinent appropriations Act.

SEC. 4. COVERAGE.

    Appropriations and funds made available, and authority granted, for 
any program, project, or activity for any fiscal year pursuant to this 
Act shall cover all obligations or expenditures incurred for that 
program, project, or activity during the portion of any fiscal year for 
which this Act applies to that program, project, or activity.

SEC. 5. EXPENDITURES.

    Expenditures made for a program, project, or activity for any 
fiscal year pursuant to this Act shall be charged to the applicable 
appropriation, fund, or authorization whenever a regular appropriation 
bill or a joint resolution making continuing appropriations until the 
end of any fiscal year providing for that program, project, or activity 
for that period becomes law.

SEC. 6. INITIATING OR RESUMING A PROGRAM, PROJECT, OR ACTIVITY.

    No appropriation or funds made available or authority granted 
pursuant to this Act shall be used to initiate or resume any program, 
project, or activity for which appropriations, funds, or other 
authority were not available during fiscal year 1999.

SEC. 7. PROTECTION OF OTHER OBLIGATIONS.

    Nothing in this Act shall be construed to effect Government 
obligations mandated by other law, including obligations with respect 
to Social Security, Medicare, Medicaid, and veterans benefits.

SEC. 8. DEFINITION.

    In this Act, the term ``regular appropriation bill'' means any 
annual appropriation bill making appropriations, otherwise making funds 
available, or granting authority, for any of the following categories 
of programs, projects, and activities:
            (1) Agriculture, rural development, and related agencies 
        programs.
            (2) The Departments of Commerce, Justice, and State, the 
        judiciary, and related agencies.
            (3) The Department of Defense.
            (4) The government of the District of Columbia and other 
        activities chargeable in whole or in part against the revenues 
        of the District.
            (5) The Departments of Labor, Health and Human Services, 
        and Education, and related agencies.
            (6) The Departments of Veterans Affairs and Housing and 
        Urban Development, and sundry independent agencies, boards, 
        commissions, corporations, and offices.
            (7) Energy and water development.
            (8) Foreign assistance and related programs.
            (9) The Department of the Interior and related agencies.
            (10) Military construction.
            (11) The Department of Transportation and related agencies.
            (12) The Treasury Department, the U.S. Postal Service, the 
        Executive Office of the President, and certain independent 
        agencies.
            (13) The legislative branch.
Summary:"""

In [10]:
# Prompt (match your training format!)

model.to("cuda")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#inputs = torch.tensor(train_ds[0]['input_ids'])[None,:].to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        #inputs,
        max_new_tokens=100,
        do_sample=True,       # or False for greedy
        temperature=0.7,
        top_p=0.9,
        #use_cache = False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1432 > 512). Running this sequence through the model will result in indexing errors


incremental_state [first run KILL ME]
incremental_state [first run KILL ME]
incremental_state [first run KILL ME]
incremental_state [first run KILL ME]
incremental_state [first run KILL ME]
incremental_state [first run KILL ME]
incremental_state [first run KILL ME]
incremental_state [first run KILL ME]
SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention 
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal 
year does not become law prior to the beginning of these fiscal years 
or a joint resolution making continuing appropriations is not in 
effect, there is appropriated, out of any moneys in the Treasury not 
otherwise appropriated, and out of applicable corporate or other 
revenues, receipts, and funds, such sums as may be necessary to 
continue any program, project, or activity for which funds were 
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made avai

In [11]:
# SANITY CHECK CELL
import importlib
import sys
import retnet_hf
importlib.reload(sys.modules['retnet_hf'])
importlib.reload(retnet_hf)
# optional but safer if things are really stuck
sys.modules["retnet_hf"] = retnet_hf

del model
del trainer

model = retnet_hf.RetNetForCausalLM(config)
model.resize_token_embeddings(len(tokenizer))


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)
train_result = trainer.train("outputs/retnet-billsum/checkpoint-8567")


model.to("cuda")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        #inputs,
        max_new_tokens=100,
        do_sample=True,       # or False for greedy
        temperature=0.7,
        top_p=0.9,
        #use_cache = False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

[transformers] Generate config GenerationConfig {
  "bos_token_id": 50256,
  "eos_token_id": 50256,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 50256,
  "use_cache": false
}

[transformers] You are resizing the embedding layer without providing a `pad_to_multiple_of` parameter. This means that the new embedding dimension will be 50257. This might induce some performance reduction as *Tensor Cores* will not be available. For more details about this, or help on choosing the correct value for resizing, refer to this guide: https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html#requirements-tc
[transformers] Loading model from outputs/retnet-billsum/checkpoint-8567.
[transformers] There were missing keys in the checkpoint model loaded: ['model.decoder.embed_tokens.weight', 'model.decoder.output_projection.weight', 'lm_head.weight'].
[transformers] ***** Running training *****
[transformers]   Num examples = 1,51

Step,Training Loss,Validation Loss


SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention 
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal 
year does not become law prior to the beginning of these fiscal years 
or a joint resolution making continuing appropriations is not in 
effect, there is appropriated, out of any moneys in the Treasury not 
otherwise appropriated, and out of applicable corporate or other 
revenues, receipts, and funds, such sums as may be necessary to 
continue any program, project, or activity for which funds were 
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made available, and 
authority granted, for a program, project, or activity for any fiscal 
year pursuant to this Act shall be at 100 percent of the rate of 
operations that was provided for the program, project, or activity in 
fiscal year 1999 in the corresponding regular appropriation Act for 
fiscal year 199